<div dir="rtl" align="right">

# KNN بِتَغييرِ k

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُدرّبُ مُصنّفاتِ أقربِ جارٍ K بِقيمِ k مُختلفةٍ ونُقارنُ دقّتَها على سماتِ قُوّةِ النطاقِ.

## ماذا يَعمَلُ هذا الدفترُ؟

يَحسبُ سماتِ قُوّةِ النطاقِ، ويُقسّمُ إلى تدريب/اختبار، ويُقيّسُ، ويُقيّمُ KNN لِـ k = 1, 3, 5, 7, 9, 11, 15, 21.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ خطّيٌّ لِدقّةِ الاختبارِ مقابلَ قيمةِ k
- مصفوفةُ الالتباسِ لِأفضلِ k
- الدقّةُ مطبوعةٌ لِكلِّ قيمةِ k

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| k values | 1, 3, 5, 7, 9, 11, 15, 21 |
| test_size | 0.2 |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تدريبُ KNN بِتَغييرِ k

</div>


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

K_VALUES = [1, 3, 5, 7, 9, 11, 15, 21]
accuracies = []
best_k = K_VALUES[0]
best_acc = 0.0
best_cm = None

for k in K_VALUES:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f'k={k}: accuracy={acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        best_k = k
        best_cm = confusion_matrix(y_test, y_pred, labels=['left_hand', 'right_hand'])

print(f'Best k: {best_k}, accuracy: {best_acc:.4f}')
classes = ['left_hand', 'right_hand']


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- k صغيرٌ جداً (مثل k=1) قد يُفرطُ في التخصيصِ، مُعطياً تبايناً عالياً
- k كبيرٌ يُنعّمُ حدَّ القرارِ لكنّهُ قد يُفرطُ في التعميمِ
- أفضلُ k يُوازنُ بينَ الانحيازِ والتباينِ

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Accuracy vs k Value',
    f'Confusion Matrix (best k={{best_k}})'))

fig.add_trace(go.Scatter(x=K_VALUES, y=accuracies, mode='lines+markers',
    marker_size=10, line_color='steelblue', name='Accuracy'), row=1, col=1)
fig.add_trace(go.Heatmap(z=best_cm, x=classes, y=classes, colorscale='Blues',
    text=best_cm, texttemplate='%{text}', textfont={'size': 16}, name='CM', showscale=True), row=2, col=1)

fig.update_xaxes(title_text='k value', row=1, col=1)
fig.update_yaxes(title_text='Test Accuracy', row=1, col=1)
fig.update_xaxes(title_text='Predicted', row=2, col=1)
fig.update_yaxes(title_text='True', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='KNN Classification - Effect of k')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- KNN حسّاسٌ لِاختيارِ k: الصغيرُ جداً يُفرطُ في التخصيصِ، والكبيرُ جداً يُفرطُ في التعميمِ
- تقييسُ السماتِ ضروريٌّ لِـ KNN لأنّهُ يَعتمدُ على حسابِ المسافةِ
- منحنى الدقّةِ مقابلَ k يُساعدُ على تحديدِ حجمِ الجوارِ الأمثلِ

</div>
